# QAOA MaxCut on C4

PennyLane QAOA with autograd: `qml.ApproxTimeEvolution` for cost/mixer layers, COBYLA optimiser, parameter-shift gradients.

In [ ]:
import numpy as np
import pennylane as qml
import scipy.optimize as opt

## Setup

In [ ]:
N_QUBITS = 4
LAYERS = 2
EDGES = [(0, 1), (1, 2), (2, 3), (3, 0)]
dev = qml.device("default.qubit", wires=N_QUBITS)

def maxcut_cost_hamiltonian(edges):
    coeffs, ops = [], []
    for i, j in edges:
        coeffs.append(0.5)
        ops.append(qml.Z(i) @ qml.Z(j))
        coeffs.append(-0.5)
        ops.append(qml.Identity(i) @ qml.Identity(j))
    return qml.Hamiltonian(coeffs, ops)

def mixer_hamiltonian(n_qubits):
    return qml.Hamiltonian(
        [1.0] * n_qubits,
        [qml.X(i) for i in range(n_qubits)],
    )

cost_ham = maxcut_cost_hamiltonian(EDGES)
mixer_ham = mixer_hamiltonian(N_QUBITS)

## QAOA circuit

In [ ]:
@qml.qnode(dev, diff_method="parameter-shift")
def qaoa_circuit(params):
    p = len(params) // 2
    gammas, betas = params[:p], params[p:]
    qml.Hadamard(wires=range(N_QUBITS))
    for k in range(p):
        qml.ApproxTimeEvolution(cost_ham, gammas[k], 1)
        qml.ApproxTimeEvolution(mixer_ham, betas[k], 1)
    return qml.expval(cost_ham)

def cut_value(bitstring):
    return sum(bitstring[i] != bitstring[j] for i, j in EDGES)

def cost_function(params):
    return -qaoa_circuit(params)

print(qml.draw(qaoa_circuit)(np.zeros(2 * LAYERS)))

## Optimise

In [ ]:
rng = np.random.default_rng(42)
init = rng.uniform(0, np.pi, size=2 * LAYERS)

result = opt.minimize(cost_function, init, method="COBYLA",
                      options={"maxiter": 100, "rhobeg": 0.4})

print(f"Converged: {result.success}")
print(f"Optimal params: {np.round(result.x, 4)}")

## Results

In [ ]:
@qml.qnode(dev)
def probabilities(params):
    p = len(params) // 2
    gammas, betas = params[:p], params[p:]
    qml.Hadamard(wires=range(N_QUBITS))
    for k in range(p):
        qml.ApproxTimeEvolution(cost_ham, gammas[k], 1)
        qml.ApproxTimeEvolution(mixer_ham, betas[k], 1)
    return qml.probs(wires=range(N_QUBITS))

probs = probabilities(result.x)
best_idx = np.argmax(probs)
best_bits = format(best_idx, f"0{N_QUBITS}b")
print(f"Most likely: |{best_bits}⟩  cut={cut_value(best_bits)}  P={probs[best_idx]:.4f}")

top = np.argsort(probs)[-4:][::-1]
print("\nTop 4 outcomes:")
for idx in top:
    bits = format(idx, f"0{N_QUBITS}b")
    print(f"  |{bits}⟩  cut={cut_value(bits)}  P={probs[idx]:.4f}")